# 02 — Baseline Model

## Business case

Recap from `01_eda.ipynb`: the target (`target`, ~5.6% positive) marks whether a client
committed electricity/gas fraud. The operational goal is a ranked shortlist for a
capacity-limited inspection team, so **PR-AUC (average precision)** is the primary metric, with
precision@k/recall@k reported for the actual inspection budget.

## What this notebook does

Compares a `DummyClassifier` floor against a first real classifier — **logistic regression**,
not literal linear regression, since the target is a 0/1 label rather than a continuous
quantity, and PR-AUC/ranking quality is what logistic regression is built to optimize (this
matches the README's own modelling guardrail and the worksheet, which both name logistic
regression as the first model to try).

It loads the already-cleaned, client-level feature table `01_eda.ipynb` now saves
(`data/processed/train_client_features.csv`) — Step 8 of that notebook already applied the
EDA's own data-quality decisions (negative-tenure fix, `months_number` outlier cap), so this
notebook only adds the generic ML preprocessing (impute/scale/encode) that belongs in every
model notebook, without repeating any of the EDA's cleaning logic.

Per the README's milestone split, this notebook only uses the **development** clients
(`train_client_features.csv`); the held-out test clients are scored once, later, in
`04_final_model.ipynb`.

## Imports and constants

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
TRAIN_FEATURES_PATH = PROCESSED_DIR / "train_client_features.csv"

RSEED = 42  # same seed 01_eda.ipynb uses for its train/test split
N_SPLITS = 5
TOP_K_SHARE = 0.10  # matches the precision@10%/recall@10% framing already used in the worksheet

if not TRAIN_FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"Missing {TRAIN_FEATURES_PATH}. Run 01_eda.ipynb with SAVE_FEATURES = True first "
        "to produce the cleaned, client-level feature table this notebook loads."
    )

## Step 1 — Load the cleaned, client-level features

In [ ]:
df = pd.read_csv(TRAIN_FEATURES_PATH)
df["client_id"] = df["client_id"].astype("string")
for column in ["creation_date", "first_invoice", "last_invoice"]:
    df[column] = pd.to_datetime(df[column])
df["target"] = df["target"].astype("int8")
df["has_negative_tenure"] = df["has_negative_tenure"].astype("int8")

overview = pd.DataFrame({
    "rows": [len(df)],
    "unique_clients": [df["client_id"].nunique()],
    "fraud_rate": [df["target"].mean()],
})
display(overview)
display(df.head())

## Step 2 — Preprocessing

Generic ML plumbing only — the business-logic cleaning already happened once in
`01_eda.ipynb`'s Step 8:

- **Excluded from the feature matrix** (kept in `df` for reference, just not modelled):
  `client_id` (key, never a predictor per the README), the raw `creation_date`/`first_invoice`/
  `last_invoice` dates (Step 4's decision — their signal already lives in derived columns like
  `active_days`/`customer_tenure_days_at_last_invoice`), and `invoice_count_group` (an EDA-only
  bin label from Step 3, not a real feature).
- **Numeric columns**: median-impute (several columns are `NaN` for clients with too little
  invoice/meter history to compute a rate, e.g. `mean_submission_index_delta`), then
  standard-scale — logistic regression's regularisation only treats features comparably once
  they're on the same scale.
- **Categorical columns** (`client_catg`, `region`, `disrict` — small cardinality, 3/25/4
  distinct codes): one-hot encode with `handle_unknown="ignore"` so an unseen code in a fold
  doesn't crash the pipeline.
- All of this is wrapped in one `ColumnTransformer` inside a `Pipeline`, so it is fit fresh
  inside every cross-validation fold — nothing about a validation fold leaks into imputation,
  scaling, or encoding.

In [ ]:
exclude_columns = [
    "client_id", "target", "creation_date", "first_invoice", "last_invoice", "invoice_count_group",
]
categorical_features = ["client_catg", "region", "disrict"]
numeric_features = [
    column for column in df.columns
    if column not in exclude_columns and column not in categorical_features
]

print(f"{len(numeric_features)} numeric features, {len(categorical_features)} categorical features")

X = df[numeric_features + categorical_features]
y = df["target"]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

## Step 3 — Dummy baseline vs. logistic regression (cross-validated)

`class_weight="balanced"` re-weights the loss by class frequency — a near-free correction for
the ~5.6% imbalance. This is a lighter alternative to SMOTE (discussed earlier as a possible
"advanced model" step); it's worth trying SMOTE-in-pipeline later, but not before seeing what
this costs nothing to try first.

Both models are cross-validated on the *development* clients only, using the same
`StratifiedKFold` split, and scored on `average_precision` (PR-AUC, primary) and `roc_auc`
(secondary) — matching the metric stack in the README.

In [ ]:
dummy_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", DummyClassifier(strategy="stratified", random_state=RSEED)),
])
logreg_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RSEED)),
])

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RSEED)

comparison_rows = {}
for name, pipeline in [("dummy", dummy_pipeline), ("logistic_regression", logreg_pipeline)]:
    cv_results = cross_validate(
        pipeline, X, y, cv=cv, scoring=["average_precision", "roc_auc"]
    )
    comparison_rows[name] = {
        "average_precision_mean": cv_results["test_average_precision"].mean(),
        "average_precision_std": cv_results["test_average_precision"].std(),
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_std": cv_results["test_roc_auc"].std(),
    }

model_comparison = pd.DataFrame(comparison_rows).T
display(model_comparison)

base_rate = y.mean()
print(f"Base fraud rate (dummy's expected PR-AUC floor): {base_rate:.4f}")

## Step 4 — Capacity-aware evaluation: precision@k / recall@k

Out-of-fold logistic-regression scores (one prediction per training client, each produced by a
model that never saw that client during fitting) are ranked once, then sliced at the top
`TOP_K_SHARE` — this is the same operating point the worksheet's KPI table commits to.

In [ ]:
oof_proba = cross_val_predict(logreg_pipeline, X, y, cv=cv, method="predict_proba")[:, 1]

y_values = y.to_numpy()
k = int(np.ceil(TOP_K_SHARE * len(y_values)))
ranked_order = np.argsort(oof_proba)[::-1]
top_k_idx = ranked_order[:k]

precision_at_k = y_values[top_k_idx].mean()
recall_at_k = y_values[top_k_idx].sum() / y_values.sum()
implied_recall_at_k = precision_at_k * TOP_K_SHARE / base_rate  # cross-check via the identity
                                                                 # recall@k = precision@k * k / base_rate

capacity_summary = pd.DataFrame({
    "value": [k, precision_at_k, recall_at_k, implied_recall_at_k],
}, index=[
    f"clients inspected (top {TOP_K_SHARE:.0%})",
    "precision@k", "recall@k", "recall@k (via precision@k * k / base_rate, cross-check)",
])
display(capacity_summary)

threshold_at_k = oof_proba[ranked_order[k - 1]]
y_pred_at_k = (oof_proba >= threshold_at_k).astype(int)
cm = confusion_matrix(y_values, y_pred_at_k)
ConfusionMatrixDisplay(cm, display_labels=["not fraud", "fraud"]).plot(cmap="Blues")
plt.title(f"Confusion matrix at the top-{TOP_K_SHARE:.0%} cutoff")
plt.show()

### Precision-recall curve

In [ ]:
precisions, recalls, _ = precision_recall_curve(y_values, oof_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recalls, precisions, label="Logistic regression", color="#D62828")
ax.axhline(base_rate, linestyle="--", color="grey", label=f"Dummy floor ({base_rate:.1%})")
ax.scatter([recall_at_k], [precision_at_k], color="black", zorder=5,
           label=f"Top {TOP_K_SHARE:.0%} cutoff")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-recall curve (out-of-fold)")
ax.legend()
plt.tight_layout()
plt.show()

## Results

- **Dummy floor**: PR-AUC 0.056, ROC-AUC 0.502 — as expected, an uninformative ranking scores
  PR-AUC equal to the base fraud rate (5.58%) and ROC-AUC of 0.5.
- **Logistic regression**: PR-AUC **0.173** (about **3.1x** the dummy floor) and ROC-AUC
  **0.799**, cross-validated over 5 folds on the 108,394 development clients — a real, if modest,
  first signal from ~50 client-level features and no elaborate feature engineering yet.
- **Precision@10%: 20.7%, recall@10%: 37.2%** — both clear the worksheet's KPI targets
  (≥17% precision@10%, ≥30% recall@10%) already at this first-baseline stage. The
  `recall@k = precision@k × k / base_rate` identity checked out almost exactly (0.3715 vs.
  0.3715), confirming the two numbers are internally consistent.
- Inspecting the top 10% (10,840 clients) means reviewing roughly 4x more non-fraud than fraud
  cases per catch — worth stating explicitly alongside the precision number when this goes in
  front of the stakeholder.

### Next steps

- Try a tree ensemble (e.g. random forest, gradient boosting) as a second real baseline —
  the README names this as the other first-model option alongside logistic regression, and
  it's a natural next comparison given ROC-AUC/PR-AUC still have real room to improve.
- Tune the classification threshold / `class_weight` against the actual inspection capacity
  rather than assuming top-10% is the right cutoff.
- Try SMOTE inside the pipeline (fit only on the training fold) as an alternative to
  `class_weight="balanced"`, and compare.
- Revisit the `months_number` clipping approximation from `01_eda.ipynb` Step 8 if
  `mean_months`/`mean_billing_months` turn out to matter a lot to the model.
- Carry the same preprocessing into `03_error_analysis.ipynb` to look at *which* clients the
  model misses, not just the aggregate rate.